In [ ]:
# Locate the repository when Jupyter starts in a notebook subdirectory.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents)
              if (p / "figure" / "paths.py").is_file()
              and (p / "run_cross_validation.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the cloned sAge repository.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from figure.paths import input_path, output_path, font_path


# figure-6-1-leipameisu

Analyze rapamycin intervention results.

Run Jupyter from the repository root. Required external data and results are listed in `figure/INPUTS.md`. Set `SAGE_FIGURE_INPUT_ROOT` and `SAGE_FIGURE_OUTPUT_ROOT` when using other directories. See figure/VALIDATION.md for the execution checks and their limits.


end

In [ ]:
import os
import sys
import re
import joblib
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import ttest_ind, pearsonr
import scipy.sparse as sp



sc.settings.set_figure_params(dpi=300, facecolor='white', format='pdf')


mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'sans-serif']


mpl.rcParams['font.size'] = 6
mpl.rcParams['axes.titlesize'] = 6
mpl.rcParams['axes.labelsize'] = 6
mpl.rcParams['xtick.labelsize'] = 6
mpl.rcParams['ytick.labelsize'] = 6
mpl.rcParams['legend.fontsize'] = 6


mpl.rcParams['axes.linewidth'] = 0.6
mpl.rcParams['xtick.major.width'] = 0.6
mpl.rcParams['ytick.major.width'] = 0.6
mpl.rcParams['xtick.direction'] = 'out'
mpl.rcParams['ytick.direction'] = 'out'

mpl.rcParams['pdf.fonttype'] = 42         
mpl.rcParams['ps.fonttype'] = 42


CLUSTER_COLORS = {
    'Cluster 1': '#D55E00', 
    'Cluster 2': '#0072B2', 
    'Cluster 3': '#009E73', 
    'Cluster 4': '#CC79A7', 
    'Unclustered': '#DDDDDD' 
}

COND_COLORS = {
    'YC': '#61A48F',  
    'OC': '#EA8D74',  
    'OT': '#8798C4'   
}


TARGET_TISSUE = "Large_Intestine" 

MODEL_PATH = input_path(f"2-8.3-shanda/1-feature/9-Master-raw-Master_Clocks_3/{TARGET_TISSUE}_Clock.pkl")
DATA_DIR = input_path("2-8.3-shanda/1-data/2-valid/GSE210669_RAW") 
OUTPUT_FIG_DIR = output_path("2-8.3-shanda/1-feature/1-figure/0-4-result-5-CR-leipameisu/2-leipameisui")
os.makedirs(OUTPUT_FIG_DIR, exist_ok=True)
sc.settings.figdir = OUTPUT_FIG_DIR 

CLUSTER_CSV_PATH = input_path(f"2-8.3-shanda/1-feature/10-2-0-Gene_Trajectory_Clusters_Annotated/{TARGET_TISSUE}_Cluster_Assignments.csv")

# ==========================================

# ==========================================
print(f"--- 正在加载 {TARGET_TISSUE} 模型与聚类信息 ---")
if not os.path.exists(MODEL_PATH): sys.exit(f"找不到模型文件: {MODEL_PATH}")

model_package = joblib.load(MODEL_PATH)
master_model = model_package['model']
clock_features = model_package['features']

cluster_dict = {}
if os.path.exists(CLUSTER_CSV_PATH):
    df_clusters = pd.read_csv(CLUSTER_CSV_PATH, index_col=0)
    cluster_dict = {gene: f"Cluster {int(cid) + 1}" for gene, cid in df_clusters['Cluster'].items()}
else:
    print(f"⚠️ 警告: 找不到聚类文件 {CLUSTER_CSV_PATH}")

print("\n正在加载外部验证数据集...")
h5_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.h5') and re.search(r'M[134]', f)]
adatas = []
for file in h5_files:
    file_path = os.path.join(DATA_DIR, file)
    sample_name = re.search(r'M[134][A-C]', file).group()
    adata_tmp = sc.read_10x_h5(file_path)
    adata_tmp.var_names_make_unique()
    adata_tmp.obs['SampleID'] = sample_name
    
    group_prefix = sample_name[:2]
    if group_prefix == 'M1': adata_tmp.obs['Condition'] = 'OC'
    elif group_prefix == 'M3': adata_tmp.obs['Condition'] = 'OT'
    elif group_prefix == 'M4': adata_tmp.obs['Condition'] = 'YC'
    adatas.append(adata_tmp)

adata = ad.concat(adatas, join="outer")
adata.obs_names_make_unique()

adata.var['mt'] = adata.var_names.str.startswith('mt-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
sc.pp.filter_cells(adata, min_genes=200)
adata = adata[(adata.obs['n_genes_by_counts'] > 500) & (adata.obs['pct_counts_mt'] < 10)].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.obs['Condition'] = pd.Categorical(adata.obs['Condition'], categories=['YC', 'OC', 'OT'], ordered=True)

X_target = np.zeros((adata.n_obs, len(clock_features)))
available_genes = []

for i, gene in enumerate(clock_features):
    mouse_gene = gene.capitalize()
    if mouse_gene in adata.var_names:
        expr = adata[:, mouse_gene].X
        X_target[:, i] = expr.toarray().flatten() if sp.issparse(expr) else expr.flatten()
        available_genes.append(mouse_gene)
    elif gene in adata.var_names:
        expr = adata[:, gene].X
        X_target[:, i] = adata[:, gene].X.toarray().flatten() if sp.issparse(adata[:, gene].X) else adata[:, gene].X.flatten()
        available_genes.append(gene)

adata.obs['Predicted_Age'] = master_model.predict(X_target)


# ==========================================

# ==========================================
MM2IN = 1 / 25.4
AXES_W = 42 * MM2IN
AXES_H = 38 * MM2IN

def set_closed_box(ax):
    """Use a black box frame with 0.6-point lines and no grid."""
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.6)
    ax.tick_params(direction='out', length=3.0, width=0.6, colors='black')
    ax.grid(False)

def add_stat_bracket(ax, x1, x2, y, h, p_val):
    """Draw an inverted U-shaped significance bracket."""
    ax.plot([x1, x1, x2, x2], [y-h, y, y, y-h], lw=0.6, c='black') 
    p_str = "< 2.2e-16" if p_val < 2.2e-16 else f"= {p_val:.1e}"
    stat_text = f"P {p_str}" 

    ax.text((x1+x2)*.5, y + h*0.1, stat_text, ha='center', va='bottom', color='black', fontsize=6)

# ==========================================

# ==========================================
print("\n正在生成绝对比例 (42x38mm) 的单张图表...")

# --------------------------------------------------------

# --------------------------------------------------------
l_marg, r_marg, b_marg, t_marg = 0.55, 0.2, 0.45, 0.35 
fig_w = AXES_W + l_marg + r_marg
fig_h = AXES_H + b_marg + t_marg

fig1 = plt.figure(figsize=(fig_w, fig_h))
ax1 = fig1.add_axes([l_marg/fig_w, b_marg/fig_h, AXES_W/fig_w, AXES_H/fig_h])

sns.violinplot(x='Condition', y='Predicted_Age', data=adata.obs, 
               palette=[COND_COLORS['YC'], COND_COLORS['OC'], COND_COLORS['OT']], 
               inner=None, linewidth=0.5, ax=ax1, saturation=1.0) 

sns.boxplot(x='Condition', y='Predicted_Age', data=adata.obs, 
            color='white', width=0.12, fliersize=0, zorder=2, ax=ax1, 
            boxprops={'edgecolor':'black','linewidth':0.6},
            medianprops={'color':'black', 'linewidth':0.8}, 
            whiskerprops={'color':'black', 'linewidth':0.6}, 
            capprops={'color':'black', 'linewidth':0.6})

yc_pred = adata.obs[adata.obs['Condition'] == 'YC']['Predicted_Age']
oc_pred = adata.obs[adata.obs['Condition'] == 'OC']['Predicted_Age']
ot_pred = adata.obs[adata.obs['Condition'] == 'OT']['Predicted_Age']

yc_median = yc_pred.median()
ax1.axhline(yc_median, color='grey', linestyle='--', linewidth=0.6, alpha=0.8, zorder=0) 

_, p_aging = ttest_ind(oc_pred, yc_pred, alternative='greater', equal_var=False)
_, p_treat = ttest_ind(ot_pred, oc_pred, alternative='less', equal_var=False)

y_min = adata.obs['Predicted_Age'].quantile(0.01)
y_max = adata.obs['Predicted_Age'].quantile(0.99)
h_offset = (y_max - y_min) * 0.05 

add_stat_bracket(ax1, 0, 1, y_max + h_offset*3, h_offset*1.5, p_aging)
add_stat_bracket(ax1, 1, 2, y_max + h_offset*6, h_offset*1.5, p_treat) 

ax1.set_ylim(y_min, y_max + h_offset * 11)

ax1.set_ylabel("Predicted age (months)")
ax1.set_xticklabels(['YC', 'OC', 'OT'])
ax1.set_xlabel("")

set_closed_box(ax1) 
fig1_path = os.path.join(OUTPUT_FIG_DIR, f"{TARGET_TISSUE}_Result5_Violin.pdf")
plt.savefig(fig1_path)
plt.close(fig1)

# --------------------------------------------------------

# --------------------------------------------------------
sc.tl.rank_genes_groups(adata, groupby='Condition', reference='YC', groups=['OC'], method='wilcoxon', key_added='DE_aging')
sc.tl.rank_genes_groups(adata, groupby='Condition', reference='OC', groups=['OT'], method='wilcoxon', key_added='DE_treatment')

logfc_aging = sc.get.rank_genes_groups_df(adata, group='OC', key='DE_aging').set_index('names')['logfoldchanges']
logfc_treat = sc.get.rank_genes_groups_df(adata, group='OT', key='DE_treatment').set_index('names')['logfoldchanges']

df_plot = pd.DataFrame({'logFC_Aging': logfc_aging[available_genes], 'logFC_Treatment': logfc_treat[available_genes]}).dropna()
cluster_dict_lower = {str(k).lower(): v for k, v in cluster_dict.items()}
df_plot['Cluster'] = df_plot.index.str.lower().map(cluster_dict_lower).fillna('Unclustered')

def get_sort_priority(cluster_name):
    if cluster_name == 'Unclustered': return 0
    elif cluster_name == 'Cluster 2': return 2 
    else: return 1
df_plot['Sort_Key'] = df_plot['Cluster'].apply(get_sort_priority)
df_plot = df_plot.sort_values('Sort_Key')

l_marg, r_marg, b_marg, t_marg = 0.55, 1.2, 0.45, 0.35 
fig_w = AXES_W + l_marg + r_marg
fig_h = AXES_H + b_marg + t_marg

fig2 = plt.figure(figsize=(fig_w, fig_h))
ax2 = fig2.add_axes([l_marg/fig_w, b_marg/fig_h, AXES_W/fig_w, AXES_H/fig_h])

sns.scatterplot(data=df_plot, x='logFC_Aging', y='logFC_Treatment', 
                hue='Cluster', palette=CLUSTER_COLORS, 
                alpha=0.9, s=25, edgecolor='white', linewidth=0.3, ax=ax2)

ax2.axhline(0, color='grey', linestyle='dashed', lw=0.6, zorder=0) 
ax2.axvline(0, color='grey', linestyle='dashed', lw=0.6, zorder=0) 

r, p_corr = pearsonr(df_plot['logFC_Aging'], df_plot['logFC_Treatment'])
p_corr_str = "< 2.2e-16" if p_corr < 2.2e-16 else f"= {p_corr:.1e}"

sns.regplot(x='logFC_Aging', y='logFC_Treatment', data=df_plot, scatter=False, 
            color='black', ax=ax2, line_kws={'linestyle':'--', 'lw':0.8}, ci=95, seed=42)

ax2.set_title(f"Pearson R = {r:.2f}, P {p_corr_str}", pad=8, fontweight='normal')
ax2.set_xlabel(r"Aging Effect: $\log_2$(OC / YC)")
ax2.set_ylabel(r"Treatment Effect: $\log_2$(OT / OC)")

handles, labels = ax2.get_legend_handles_labels()
ax2.legend(handles, labels, bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False, 
           handletextpad=0.1)

set_closed_box(ax2)
fig2_path = os.path.join(OUTPUT_FIG_DIR, f"{TARGET_TISSUE}_Result5_Scatter.pdf")
plt.savefig(fig2_path)
plt.close(fig2)


# ==========================================

# ==========================================
def plot_advanced_reversal_signatures(adata, valid_genes, output_dir, tissue_name, cluster_dict):
    print("正在生成独立的高级逆转特征轨迹与聚类热图...")
    df_means = pd.DataFrame(index=valid_genes, columns=['YC', 'OC', 'OT'])
    for cond in ['YC', 'OC', 'OT']:
        subset = adata[adata.obs['Condition'] == cond, valid_genes].X
        df_means[cond] = np.ravel(subset.mean(axis=0)) if sp.issparse(subset) else np.ravel(subset.mean(axis=0))

    df_zscore = df_means.apply(lambda x: (x - x.mean()) / (x.std() + 1e-8), axis=1)
    
    l_marg, r_marg, b_marg, t_marg = 0.55, 1.2, 0.45, 0.35 
    fig_w = AXES_W + l_marg + r_marg
    fig_h = AXES_H + b_marg + t_marg
    fig3 = plt.figure(figsize=(fig_w, fig_h))
    ax3 = fig3.add_axes([l_marg/fig_w, b_marg/fig_h, AXES_W/fig_w, AXES_H/fig_h])
    
    x_labels = ['YC', 'OC', 'OT']
    
    for gene in df_zscore.index:
        c_label = cluster_dict.get(gene, 'Unclustered')
        color = CLUSTER_COLORS.get(c_label, '#DDDDDD')
        alpha_val = 0.08 if c_label == 'Unclustered' else 0.2
        ax3.plot(x_labels, df_zscore.loc[gene], color=color, alpha=alpha_val, linewidth=0.4, zorder=1) 
    
    import matplotlib.patheffects as pe
    for cluster_name, color in CLUSTER_COLORS.items():
        if cluster_name == 'Unclustered': continue
        cluster_genes = [g for g in df_zscore.index if cluster_dict.get(g) == cluster_name]
        if cluster_genes:
            mean_trend = df_zscore.loc[cluster_genes].mean(axis=0)
            ax3.plot(x_labels, mean_trend, marker='o', color=color, linewidth=1.2, markersize=3, 
                     markeredgecolor='white', markeredgewidth=0.6,
                     path_effects=[pe.Stroke(linewidth=2.0, foreground='white'), pe.Normal()],
                     label=f'{cluster_name} (n={len(cluster_genes)})', zorder=5)
    
    ax3.set_title("Cluster Trajectories", pad=8)
    ax3.set_ylabel("Relative Expression (Z-score)")
    ax3.set_xlabel("Condition")
    ax3.axhline(0, color='grey', linestyle=':', lw=0.6, zorder=0) 
    
    set_closed_box(ax3)
    ax3.legend(frameon=False, loc='center left', bbox_to_anchor=(1.05, 0.5))
    
    fig3_path = os.path.join(output_dir, f"{tissue_name}_Result5_Trajectories.pdf")
    plt.savefig(fig3_path)
    plt.close(fig3)


    with mpl.rc_context({'font.size': 6, 'figure.figsize': [2.8, 3.2]}):
        valid_genes_heatmap = [g for g in valid_genes if np.var(adata[:, g].X.toarray() if sp.issparse(adata[:, g].X) else adata[:, g].X) > 0]
        
        sc.pl.matrixplot(adata, var_names=valid_genes_heatmap, groupby='Condition', 
                         dendrogram=True, cmap='vlag', standard_scale='var', 
                         vmin=0, vmax=1, 
                         colorbar_title='Scaled Expr', 
                         title="",
                         save=f"_{tissue_name}_Result5_Heatmap.pdf", show=False)

plot_advanced_reversal_signatures(adata, available_genes, OUTPUT_FIG_DIR, TARGET_TISSUE, cluster_dict)
print(f"\n✅ 完美收官！四张独立的图表已分别存入: {OUTPUT_FIG_DIR}")

In [ ]:
import os
import sys
import re
import joblib
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import ttest_ind, pearsonr
import scipy.sparse as sp



sc.settings.set_figure_params(dpi=300, facecolor='white', format='pdf')


mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'sans-serif']


mpl.rcParams['font.size'] = 6
mpl.rcParams['axes.titlesize'] = 6
mpl.rcParams['axes.labelsize'] = 6
mpl.rcParams['xtick.labelsize'] = 6
mpl.rcParams['ytick.labelsize'] = 6
mpl.rcParams['legend.fontsize'] = 6


mpl.rcParams['axes.linewidth'] = 0.6
mpl.rcParams['xtick.major.width'] = 0.6
mpl.rcParams['ytick.major.width'] = 0.6
mpl.rcParams['xtick.direction'] = 'out'
mpl.rcParams['ytick.direction'] = 'out'

mpl.rcParams['pdf.fonttype'] = 42        
mpl.rcParams['ps.fonttype'] = 42


CLUSTER_COLORS = {
    'Cluster 1': '#D55E00', 
    'Cluster 2': '#0072B2', 
    'Cluster 3': '#009E73', 
    'Cluster 4': '#CC79A7', 
    'Unclustered': '#DDDDDD' 
}

COND_COLORS = {
    'YC': '#61A48F',  
    'OC': '#EA8D74',  
    'OT': '#8798C4'   
}


TARGET_TISSUE = "Large_Intestine" 

MODEL_PATH = input_path(f"2-8.3-shanda/1-feature/9-Master-raw-Master_Clocks_3/{TARGET_TISSUE}_Clock.pkl")
DATA_DIR = input_path("2-8.3-shanda/1-data/2-valid/GSE210669_RAW") 
OUTPUT_FIG_DIR = output_path("2-8.3-shanda/1-feature/1-figure/0-4-result-5-CR-leipameisu/2.1-leipameisui")
os.makedirs(OUTPUT_FIG_DIR, exist_ok=True)
sc.settings.figdir = OUTPUT_FIG_DIR 

CLUSTER_CSV_PATH = input_path(f"2-8.3-shanda/1-feature/10-2-0-Gene_Trajectory_Clusters_Annotated/{TARGET_TISSUE}_Cluster_Assignments.csv")

# ==========================================

# ==========================================
print(f"--- 正在加载 {TARGET_TISSUE} 模型与聚类信息 ---")
if not os.path.exists(MODEL_PATH): sys.exit(f"找不到模型文件: {MODEL_PATH}")

model_package = joblib.load(MODEL_PATH)
master_model = model_package['model']
clock_features = model_package['features']

cluster_dict = {}
if os.path.exists(CLUSTER_CSV_PATH):
    df_clusters = pd.read_csv(CLUSTER_CSV_PATH, index_col=0)
    cluster_dict = {gene: f"Cluster {int(cid) + 1}" for gene, cid in df_clusters['Cluster'].items()}
else:
    print(f"⚠️ 警告: 找不到聚类文件 {CLUSTER_CSV_PATH}")

print("\n正在加载外部验证数据集...")
h5_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.h5') and re.search(r'M[134]', f)]
adatas = []
for file in h5_files:
    file_path = os.path.join(DATA_DIR, file)
    sample_name = re.search(r'M[134][A-C]', file).group()
    adata_tmp = sc.read_10x_h5(file_path)
    adata_tmp.var_names_make_unique()
    adata_tmp.obs['SampleID'] = sample_name
    
    group_prefix = sample_name[:2]
    if group_prefix == 'M1': adata_tmp.obs['Condition'] = 'OC'
    elif group_prefix == 'M3': adata_tmp.obs['Condition'] = 'OT'
    elif group_prefix == 'M4': adata_tmp.obs['Condition'] = 'YC'
    adatas.append(adata_tmp)

adata = ad.concat(adatas, join="outer")
adata.obs_names_make_unique()

adata.var['mt'] = adata.var_names.str.startswith('mt-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
sc.pp.filter_cells(adata, min_genes=200)
adata = adata[(adata.obs['n_genes_by_counts'] > 500) & (adata.obs['pct_counts_mt'] < 10)].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.obs['Condition'] = pd.Categorical(adata.obs['Condition'], categories=['YC', 'OC', 'OT'], ordered=True)

X_target = np.zeros((adata.n_obs, len(clock_features)))
available_genes = []

for i, gene in enumerate(clock_features):
    mouse_gene = gene.capitalize()
    if mouse_gene in adata.var_names:
        expr = adata[:, mouse_gene].X
        X_target[:, i] = expr.toarray().flatten() if sp.issparse(expr) else expr.flatten()
        available_genes.append(mouse_gene)
    elif gene in adata.var_names:
        expr = adata[:, gene].X
        X_target[:, i] = adata[:, gene].X.toarray().flatten() if sp.issparse(adata[:, gene].X) else adata[:, gene].X.flatten()
        available_genes.append(gene)

adata.obs['Predicted_Age'] = master_model.predict(X_target)


# ==========================================

# ==========================================
MM2IN = 1 / 25.4
AXES_W = 42 * MM2IN
AXES_H = 38 * MM2IN

def set_closed_box(ax):
    """Use a black box frame with 0.6-point lines and no grid."""
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.6)
    ax.tick_params(direction='out', length=3.0, width=0.6, colors='black')
    ax.grid(False)

def add_stat_bracket(ax, x1, x2, y, h, p_val):
    """Draw an inverted U-shaped significance bracket."""
    ax.plot([x1, x1, x2, x2], [y-h, y, y, y-h], lw=0.6, c='black') 
    p_str = "< 2.2e-16" if p_val < 2.2e-16 else f"= {p_val:.1e}"
    stat_text = f"P {p_str}" 

    ax.text((x1+x2)*.5, y + h*0.1, stat_text, ha='center', va='bottom', color='black', fontsize=6)

# ==========================================

# ==========================================
print("\n正在生成绝对比例 (42x38mm) 的单张图表...")

# --------------------------------------------------------

# --------------------------------------------------------
l_marg, r_marg, b_marg, t_marg = 0.55, 0.2, 0.45, 0.35 
fig_w = AXES_W + l_marg + r_marg
fig_h = AXES_H + b_marg + t_marg

fig1 = plt.figure(figsize=(fig_w, fig_h))
ax1 = fig1.add_axes([l_marg/fig_w, b_marg/fig_h, AXES_W/fig_w, AXES_H/fig_h])

sns.violinplot(x='Condition', y='Predicted_Age', data=adata.obs, 
               palette=[COND_COLORS['YC'], COND_COLORS['OC'], COND_COLORS['OT']], 
               inner=None, linewidth=0.5, ax=ax1, saturation=1.0) 

sns.boxplot(x='Condition', y='Predicted_Age', data=adata.obs, 
            color='white', width=0.12, fliersize=0, zorder=2, ax=ax1, 
            boxprops={'edgecolor':'black','linewidth':0.6},
            medianprops={'color':'black', 'linewidth':0.8}, 
            whiskerprops={'color':'black', 'linewidth':0.6}, 
            capprops={'color':'black', 'linewidth':0.6})

yc_pred = adata.obs[adata.obs['Condition'] == 'YC']['Predicted_Age']
oc_pred = adata.obs[adata.obs['Condition'] == 'OC']['Predicted_Age']
ot_pred = adata.obs[adata.obs['Condition'] == 'OT']['Predicted_Age']

yc_median = yc_pred.median()
ax1.axhline(yc_median, color='grey', linestyle='--', linewidth=0.6, alpha=0.8, zorder=0) 

_, p_aging = ttest_ind(oc_pred, yc_pred, alternative='greater', equal_var=False)
_, p_treat = ttest_ind(ot_pred, oc_pred, alternative='less', equal_var=False)

y_min = adata.obs['Predicted_Age'].quantile(0.01)
y_max = adata.obs['Predicted_Age'].quantile(0.99)
h_offset = (y_max - y_min) * 0.05 

add_stat_bracket(ax1, 0, 1, y_max + h_offset*3, h_offset*1.5, p_aging)
add_stat_bracket(ax1, 1, 2, y_max + h_offset*6, h_offset*1.5, p_treat) 

ax1.set_ylim(y_min, y_max + h_offset * 11)

ax1.set_ylabel("Predicted age (months)")
ax1.set_xticklabels(['YC', 'OC', 'OT'])
ax1.set_xlabel("")

set_closed_box(ax1) 
fig1_path = os.path.join(OUTPUT_FIG_DIR, f"{TARGET_TISSUE}_Result5_Violin.pdf")
plt.savefig(fig1_path)
plt.close(fig1)

# --------------------------------------------------------

# --------------------------------------------------------
sc.tl.rank_genes_groups(adata, groupby='Condition', reference='YC', groups=['OC'], method='wilcoxon', key_added='DE_aging')
sc.tl.rank_genes_groups(adata, groupby='Condition', reference='OC', groups=['OT'], method='wilcoxon', key_added='DE_treatment')

logfc_aging = sc.get.rank_genes_groups_df(adata, group='OC', key='DE_aging').set_index('names')['logfoldchanges']
logfc_treat = sc.get.rank_genes_groups_df(adata, group='OT', key='DE_treatment').set_index('names')['logfoldchanges']

df_plot = pd.DataFrame({'logFC_Aging': logfc_aging[available_genes], 'logFC_Treatment': logfc_treat[available_genes]}).dropna()
rescued = (df_plot['logFC_Aging'] * df_plot['logFC_Treatment'] < 0)


l_marg, r_marg, b_marg, t_marg = 0.55, 0.2, 0.45, 0.35 
fig_w = AXES_W + l_marg + r_marg
fig_h = AXES_H + b_marg + t_marg

fig2 = plt.figure(figsize=(fig_w, fig_h))
ax2 = fig2.add_axes([l_marg/fig_w, b_marg/fig_h, AXES_W/fig_w, AXES_H/fig_h])


colors = ['#D95F02' if r else 'grey' for r in rescued]
ax2.scatter(df_plot['logFC_Aging'], df_plot['logFC_Treatment'], c=colors, alpha=0.9, s=15, edgecolor='white', linewidth=0.3)

ax2.axhline(0, color='grey', linestyle='dashed', lw=0.6, zorder=0) 
ax2.axvline(0, color='grey', linestyle='dashed', lw=0.6, zorder=0) 

r, p_corr = pearsonr(df_plot['logFC_Aging'], df_plot['logFC_Treatment'])
p_corr_str = "< 2.2e-16" if p_corr < 2.2e-16 else f"= {p_corr:.1e}"

sns.regplot(x='logFC_Aging', y='logFC_Treatment', data=df_plot, scatter=False, 
            color='black', ax=ax2, line_kws={'linestyle':'--', 'lw':0.8}, ci=95, seed=42)

ax2.set_title(f"Pearson R = {r:.2f}, P {p_corr_str}", pad=8, fontweight='normal')
ax2.set_xlabel(r"Aging Effect: $\log_2$(OC / YC)")
ax2.set_ylabel(r"Treatment Effect: $\log_2$(OT / OC)")

set_closed_box(ax2)
fig2_path = os.path.join(OUTPUT_FIG_DIR, f"{TARGET_TISSUE}_Result5_Scatter.pdf")
plt.savefig(fig2_path)
plt.close(fig2)


# ==========================================

# ==========================================
def plot_advanced_reversal_signatures(adata, valid_genes, output_dir, tissue_name, cluster_dict):
    print("正在生成独立的高级逆转特征轨迹与聚类热图...")
    df_means = pd.DataFrame(index=valid_genes, columns=['YC', 'OC', 'OT'])
    for cond in ['YC', 'OC', 'OT']:
        subset = adata[adata.obs['Condition'] == cond, valid_genes].X
        df_means[cond] = np.ravel(subset.mean(axis=0)) if sp.issparse(subset) else np.ravel(subset.mean(axis=0))

    df_zscore = df_means.apply(lambda x: (x - x.mean()) / (x.std() + 1e-8), axis=1)
    
    l_marg, r_marg, b_marg, t_marg = 0.55, 1.2, 0.45, 0.35 
    fig_w = AXES_W + l_marg + r_marg
    fig_h = AXES_H + b_marg + t_marg
    fig3 = plt.figure(figsize=(fig_w, fig_h))
    ax3 = fig3.add_axes([l_marg/fig_w, b_marg/fig_h, AXES_W/fig_w, AXES_H/fig_h])
    
    x_labels = ['YC', 'OC', 'OT']
    
    for gene in df_zscore.index:
        c_label = cluster_dict.get(gene, 'Unclustered')
        color = CLUSTER_COLORS.get(c_label, '#DDDDDD')
        alpha_val = 0.08 if c_label == 'Unclustered' else 0.2
        ax3.plot(x_labels, df_zscore.loc[gene], color=color, alpha=alpha_val, linewidth=0.4, zorder=1) 
    
    import matplotlib.patheffects as pe
    for cluster_name, color in CLUSTER_COLORS.items():
        if cluster_name == 'Unclustered': continue
        cluster_genes = [g for g in df_zscore.index if cluster_dict.get(g) == cluster_name]
        if cluster_genes:
            mean_trend = df_zscore.loc[cluster_genes].mean(axis=0)
            ax3.plot(x_labels, mean_trend, marker='o', color=color, linewidth=1.2, markersize=3, 
                     markeredgecolor='white', markeredgewidth=0.6,
                     path_effects=[pe.Stroke(linewidth=2.0, foreground='white'), pe.Normal()],
                     label=f'{cluster_name} (n={len(cluster_genes)})', zorder=5)
    
    ax3.set_title("Cluster Trajectories", pad=8)
    ax3.set_ylabel("Relative Expression (Z-score)")
    ax3.set_xlabel("Condition")
    ax3.axhline(0, color='grey', linestyle=':', lw=0.6, zorder=0) 
    
    set_closed_box(ax3)
    ax3.legend(frameon=False, loc='center left', bbox_to_anchor=(1.05, 0.5))
    
    fig3_path = os.path.join(output_dir, f"{tissue_name}_Result5_Trajectories.pdf")
    plt.savefig(fig3_path)
    plt.close(fig3)


    with mpl.rc_context({'font.size': 6, 'figure.figsize': [2.8, 3.2]}):
        valid_genes_heatmap = [g for g in valid_genes if np.var(adata[:, g].X.toarray() if sp.issparse(adata[:, g].X) else adata[:, g].X) > 0]
        
        sc.pl.matrixplot(adata, var_names=valid_genes_heatmap, groupby='Condition', 
                         dendrogram=True, cmap='vlag', standard_scale='var', 
                         vmin=0, vmax=1, 
                         colorbar_title='Scaled Expr', 
                         title="",
                         save=f"_{tissue_name}_Result5_Heatmap.pdf", show=False)

plot_advanced_reversal_signatures(adata, available_genes, OUTPUT_FIG_DIR, TARGET_TISSUE, cluster_dict)
print(f"\n✅ 完美收官！四张独立的图表已分别存入: {OUTPUT_FIG_DIR}")

In [ ]:
# ============================================================




#   Aging: OC > YC; Treatment: OT < OC
# ============================================================

import os
import re
import joblib
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import scipy.sparse as sp
from scipy.stats import ttest_ind, pearsonr


sc.settings.set_figure_params(dpi=300, facecolor="white", format="pdf")

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "sans-serif"]
mpl.rcParams["font.size"] = 6
mpl.rcParams["axes.titlesize"] = 6
mpl.rcParams["axes.labelsize"] = 6
mpl.rcParams["xtick.labelsize"] = 6
mpl.rcParams["ytick.labelsize"] = 6
mpl.rcParams["legend.fontsize"] = 6
mpl.rcParams["axes.linewidth"] = 0.6
mpl.rcParams["xtick.major.width"] = 0.6
mpl.rcParams["ytick.major.width"] = 0.6
mpl.rcParams["xtick.direction"] = "out"
mpl.rcParams["ytick.direction"] = "out"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

COND_COLORS = {
    "YC": "#61A48F",
    "OC": "#EA8D74",
    "OT": "#8798C4",
}

CLUSTER_COLORS = {
    "Cluster 1": "#D55E00",
    "Cluster 2": "#0072B2",
    "Cluster 3": "#009E73",
    "Cluster 4": "#CC79A7",
    "Unclustered": "#DDDDDD",
}


TARGET_TISSUE = "Large_Intestine"
MODEL_PATH = input_path(f"2-8.3-shanda/1-feature/9-Master-raw-Master_Clocks_3/{TARGET_TISSUE}_Clock.pkl")
DATA_DIR = input_path("2-8.3-shanda/1-data/2-valid/GSE210669_RAW")
CLUSTER_CSV_PATH = input_path(f"2-8.3-shanda/1-feature/10-2-0-Gene_Trajectory_Clusters_Annotated/{TARGET_TISSUE}_Cluster_Assignments.csv")

OUTPUT_FIG_DIR = output_path("2-8.3-shanda/1-feature/1-figure/0-4-result-5-CR-leipameisu/3-leipameisui/Large_Intestine_clock_to_rapamycin")
os.makedirs(OUTPUT_FIG_DIR, exist_ok=True)
sc.settings.figdir = OUTPUT_FIG_DIR


def sparse_to_1d(x):
    return x.toarray().ravel() if sp.issparse(x) else np.asarray(x).ravel()


def sparse_mean_1d(x):
    return np.asarray(x.mean(axis=0)).ravel() if sp.issparse(x) else np.asarray(x).mean(axis=0).ravel()


def set_closed_box(ax):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.6)
    ax.tick_params(direction="out", length=3.0, width=0.6, colors="black")
    ax.grid(False)


def format_p_value(p_val):
    if not np.isfinite(p_val):
        return "= NA"
    return "< 2.2e-16" if p_val < 2.2e-16 else f"= {p_val:.1e}"


def add_stat_bracket(ax, x1, x2, y, h, p_val):
    ax.plot([x1, x1, x2, x2], [y - h, y, y, y - h], lw=0.6, c="black")
    ax.text(
        (x1 + x2) * 0.5,
        y + h * 0.1,
        f"P {format_p_value(p_val)}",
        ha="center",
        va="bottom",
        color="black",
        fontsize=6,
    )


def load_cluster_dict(cluster_csv_path):
    if not os.path.exists(cluster_csv_path):
        print(f"Warning: cluster file not found: {cluster_csv_path}")
        return {}
    df_clusters = pd.read_csv(cluster_csv_path, index_col=0)
    if "Cluster" not in df_clusters.columns:
        print(f"Warning: Cluster column not found in: {cluster_csv_path}")
        return {}
    return {gene: f"Cluster {int(cid) + 1}" for gene, cid in df_clusters["Cluster"].items()}


def load_rapamycin_adata(data_dir):
    condition_map = {
        "M1": "OC",  # old control
        "M3": "OT",  # old + rapamycin treatment
        "M4": "YC",  # young control
    }

    h5_files = sorted(
        f for f in os.listdir(data_dir)
        if f.endswith(".h5") and re.search(r"M[134][A-C]", f)
    )
    if not h5_files:
        raise FileNotFoundError(f"No M1/M3/M4 h5 files found in: {data_dir}")

    adatas = []
    for filename in h5_files:
        sample_match = re.search(r"M[134][A-C]", filename)
        if sample_match is None:
            continue
        sample_id = sample_match.group()
        group_id = sample_id[:2]
        condition = condition_map[group_id]

        file_path = os.path.join(data_dir, filename)
        adata_tmp = sc.read_10x_h5(file_path)
        adata_tmp.var_names_make_unique()
        adata_tmp.obs["SampleID"] = sample_id
        adata_tmp.obs["Condition"] = condition
        adatas.append(adata_tmp)
        print(f"Loaded {sample_id}: {condition}, n_cells={adata_tmp.n_obs}")

    adata = ad.concat(adatas, join="outer", label="batch", keys=[a.obs["SampleID"].iloc[0] for a in adatas])
    adata.obs_names_make_unique()
    return adata


def preprocess_rapamycin_adata(adata):
    adata.var["mt"] = adata.var_names.str.startswith("mt-")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)
    sc.pp.filter_cells(adata, min_genes=200)
    adata = adata[(adata.obs["n_genes_by_counts"] > 500) & (adata.obs["pct_counts_mt"] < 10)].copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.obs["Condition"] = pd.Categorical(adata.obs["Condition"], categories=["YC", "OC", "OT"], ordered=True)
    return adata


def build_external_feature_matrix(adata, clock_features):
    """Build the external expression matrix in clock-feature order, filling missing genes with zero."""
    X_target = np.zeros((adata.n_obs, len(clock_features)), dtype=np.float32)
    matched_rows = []
    available_genes = []

    var_names = set(map(str, adata.var_names))
    lower_to_var = {str(g).lower(): str(g) for g in adata.var_names}

    for i, gene in enumerate(clock_features):
        candidates = [str(gene), str(gene).capitalize()]
        matched_gene = None
        for cand in candidates:
            if cand in var_names:
                matched_gene = cand
                break
        if matched_gene is None:
            matched_gene = lower_to_var.get(str(gene).lower())

        if matched_gene is not None:
            expr = adata[:, matched_gene].X
            X_target[:, i] = sparse_to_1d(expr)
            available_genes.append(matched_gene)
            matched_rows.append({"clock_gene": gene, "external_gene": matched_gene, "matched": True})
        else:
            matched_rows.append({"clock_gene": gene, "external_gene": "", "matched": False})

    match_df = pd.DataFrame(matched_rows)
    return X_target, available_genes, match_df


print(f"--- Loading {TARGET_TISSUE} aging clock ---")
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Clock model not found: {MODEL_PATH}")

model_package = joblib.load(MODEL_PATH)
master_model = model_package["model"]
clock_features = list(model_package["features"])
cluster_dict = load_cluster_dict(CLUSTER_CSV_PATH)

print("--- Loading rapamycin external dataset ---")
adata = load_rapamycin_adata(DATA_DIR)
adata = preprocess_rapamycin_adata(adata)

X_target, available_genes, match_df = build_external_feature_matrix(adata, clock_features)
adata.obs["Predicted_Age"] = master_model.predict(X_target)

match_df.to_csv(os.path.join(OUTPUT_FIG_DIR, f"{TARGET_TISSUE}_clock_gene_match.csv"), index=False)
pred_df = adata.obs[["SampleID", "Condition", "Predicted_Age"]].copy()
pred_df.to_csv(os.path.join(OUTPUT_FIG_DIR, f"{TARGET_TISSUE}_rapamycin_predicted_age_cell_level.csv"), index=True)

print(
    f"Clock genes matched: {match_df['matched'].sum()}/{len(match_df)} "
    f"({match_df['matched'].mean() * 100:.1f}%)"
)


yc_pred = adata.obs.loc[adata.obs["Condition"] == "YC", "Predicted_Age"].astype(float)
oc_pred = adata.obs.loc[adata.obs["Condition"] == "OC", "Predicted_Age"].astype(float)
ot_pred = adata.obs.loc[adata.obs["Condition"] == "OT", "Predicted_Age"].astype(float)

_, p_aging = ttest_ind(oc_pred, yc_pred, alternative="greater", equal_var=False)
_, p_treat = ttest_ind(ot_pred, oc_pred, alternative="less", equal_var=False)

summary_rows = []
for cond in ["YC", "OC", "OT"]:
    vals = adata.obs.loc[adata.obs["Condition"] == cond, "Predicted_Age"].astype(float)
    summary_rows.append({
        "Tissue": TARGET_TISSUE,
        "Condition": cond,
        "n_cells": int(vals.shape[0]),
        "mean_predicted_age": float(vals.mean()),
        "median_predicted_age": float(vals.median()),
        "std_predicted_age": float(vals.std()),
    })

sample_summary = (
    adata.obs[["SampleID", "Condition", "Predicted_Age"]]
    .groupby(["SampleID", "Condition"], observed=True)["Predicted_Age"]
    .agg(["count", "mean", "median", "std"])
    .reset_index()
)
sample_summary.to_csv(os.path.join(OUTPUT_FIG_DIR, f"{TARGET_TISSUE}_rapamycin_predicted_age_sample_summary.csv"), index=False)

stats_df = pd.DataFrame(summary_rows)
stats_df["p_aging_OC_greater_YC"] = p_aging
stats_df["p_treatment_OT_less_OC"] = p_treat
stats_df["matched_clock_genes"] = int(match_df["matched"].sum())
stats_df["total_clock_genes"] = int(len(match_df))
stats_df["clock_gene_match_rate"] = float(match_df["matched"].mean())
stats_df.to_csv(os.path.join(OUTPUT_FIG_DIR, f"{TARGET_TISSUE}_rapamycin_transfer_summary.csv"), index=False)

print(stats_df)
print(f"Aging test OC > YC: P {format_p_value(p_aging)}")
print(f"Rapamycin test OT < OC: P {format_p_value(p_treat)}")


MM2IN = 1 / 25.4
AXES_W = 42 * MM2IN
AXES_H = 38 * MM2IN
l_marg, r_marg, b_marg, t_marg = 0.55, 0.2, 0.45, 0.35
fig_w = AXES_W + l_marg + r_marg
fig_h = AXES_H + b_marg + t_marg

fig1 = plt.figure(figsize=(fig_w, fig_h))
ax1 = fig1.add_axes([l_marg / fig_w, b_marg / fig_h, AXES_W / fig_w, AXES_H / fig_h])

sns.violinplot(
    x="Condition",
    y="Predicted_Age",
    data=adata.obs,
    order=["YC", "OC", "OT"],
    palette=[COND_COLORS["YC"], COND_COLORS["OC"], COND_COLORS["OT"]],
    inner=None,
    linewidth=0.5,
    ax=ax1,
    saturation=1.0,
)
sns.boxplot(
    x="Condition",
    y="Predicted_Age",
    data=adata.obs,
    order=["YC", "OC", "OT"],
    color="white",
    width=0.12,
    fliersize=0,
    zorder=2,
    ax=ax1,
    boxprops={"edgecolor": "black", "linewidth": 0.6},
    medianprops={"color": "black", "linewidth": 0.8},
    whiskerprops={"color": "black", "linewidth": 0.6},
    capprops={"color": "black", "linewidth": 0.6},
)

ax1.axhline(float(yc_pred.median()), color="grey", linestyle="--", linewidth=0.6, alpha=0.8, zorder=0)
y_min = float(adata.obs["Predicted_Age"].quantile(0.01))
y_max = float(adata.obs["Predicted_Age"].quantile(0.99))
h_offset = max((y_max - y_min) * 0.05, 1e-6)
add_stat_bracket(ax1, 0, 1, y_max + h_offset * 3, h_offset * 1.5, p_aging)
add_stat_bracket(ax1, 1, 2, y_max + h_offset * 6, h_offset * 1.5, p_treat)
ax1.set_ylim(y_min, y_max + h_offset * 11)
ax1.set_ylabel("Predicted age (months)")
ax1.set_xlabel("")
ax1.set_xticklabels(["YC", "OC", "OT"])
set_closed_box(ax1)

fig1_path = os.path.join(OUTPUT_FIG_DIR, f"{TARGET_TISSUE}_Rapamycin_Predicted_Age_Violin.pdf")
plt.savefig(fig1_path)
plt.close(fig1)


valid_genes = [g for g in available_genes if g in adata.var_names]
if len(valid_genes) >= 2:
    sc.tl.rank_genes_groups(adata, groupby="Condition", reference="YC", groups=["OC"], method="wilcoxon", key_added="DE_aging")
    sc.tl.rank_genes_groups(adata, groupby="Condition", reference="OC", groups=["OT"], method="wilcoxon", key_added="DE_treatment")

    logfc_aging = sc.get.rank_genes_groups_df(adata, group="OC", key="DE_aging").set_index("names")["logfoldchanges"]
    logfc_treat = sc.get.rank_genes_groups_df(adata, group="OT", key="DE_treatment").set_index("names")["logfoldchanges"]

    df_plot = pd.DataFrame({
        "logFC_Aging": logfc_aging.reindex(valid_genes),
        "logFC_Treatment": logfc_treat.reindex(valid_genes),
    }).dropna()

    cluster_dict_lower = {str(k).lower(): v for k, v in cluster_dict.items()}
    df_plot["Cluster"] = df_plot.index.str.lower().map(cluster_dict_lower).fillna("Unclustered")
    df_plot["Sort_Key"] = df_plot["Cluster"].map(lambda x: 0 if x == "Unclustered" else (2 if x == "Cluster 2" else 1))
    df_plot = df_plot.sort_values("Sort_Key")
    df_plot.to_csv(os.path.join(OUTPUT_FIG_DIR, f"{TARGET_TISSUE}_Rapamycin_LogFC_Reversal.csv"))

    l_marg, r_marg, b_marg, t_marg = 0.55, 1.2, 0.45, 0.35
    fig_w = AXES_W + l_marg + r_marg
    fig_h = AXES_H + b_marg + t_marg
    fig2 = plt.figure(figsize=(fig_w, fig_h))
    ax2 = fig2.add_axes([l_marg / fig_w, b_marg / fig_h, AXES_W / fig_w, AXES_H / fig_h])

    sns.scatterplot(
        data=df_plot,
        x="logFC_Aging",
        y="logFC_Treatment",
        hue="Cluster",
        palette=CLUSTER_COLORS,
        alpha=0.9,
        s=25,
        edgecolor="white",
        linewidth=0.3,
        ax=ax2,
    )
    ax2.axhline(0, color="grey", linestyle="dashed", lw=0.6, zorder=0)
    ax2.axvline(0, color="grey", linestyle="dashed", lw=0.6, zorder=0)

    if df_plot.shape[0] >= 3:
        r, p_corr = pearsonr(df_plot["logFC_Aging"], df_plot["logFC_Treatment"])
        sns.regplot(
            x="logFC_Aging",
            y="logFC_Treatment",
            data=df_plot,
            scatter=False,
            color="black",
            ax=ax2,
            line_kws={"linestyle": "--", "lw": 0.8},
            ci=95,
            seed=42,
        )
        ax2.set_title(f"Pearson R = {r:.2f}, P {format_p_value(p_corr)}", pad=8, fontweight="normal")

    ax2.set_xlabel(r"Aging Effect: $\log_2$(OC / YC)")
    ax2.set_ylabel(r"Treatment Effect: $\log_2$(OT / OC)")
    handles, labels = ax2.get_legend_handles_labels()
    ax2.legend(handles, labels, bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False, handletextpad=0.1)
    set_closed_box(ax2)

    fig2_path = os.path.join(OUTPUT_FIG_DIR, f"{TARGET_TISSUE}_Rapamycin_LogFC_Reversal_Scatter.pdf")
    plt.savefig(fig2_path)
    plt.close(fig2)


if len(valid_genes) >= 2:
    valid_genes_heatmap = []
    for gene in valid_genes:
        x = adata[:, gene].X
        vals = x.toarray().ravel() if sp.issparse(x) else np.asarray(x).ravel()
        if np.nanvar(vals) > 0:
            valid_genes_heatmap.append(gene)

    if len(valid_genes_heatmap) >= 2:
        with mpl.rc_context({"font.size": 6, "figure.figsize": [2.8, 3.2]}):
            sc.pl.matrixplot(
                adata,
                var_names=valid_genes_heatmap,
                groupby="Condition",
                dendrogram=True,
                cmap="vlag",
                standard_scale="var",
                vmin=0,
                vmax=1,
                colorbar_title="Scaled Expr",
                title="",
                save=f"_{TARGET_TISSUE}_Rapamycin_Clock_Gene_Heatmap.pdf",
                show=False,
            )

print(f"Done. Results saved to: {OUTPUT_FIG_DIR}")
